# Loading and Inspecting Medical Images

This notebook demonstrates how to load DICOM and PNG images using the `medical_image` framework.

In [1]:
!pip install medical-image-std

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 69.6 MB/s eta 0:00:00


In [4]:
!wget -O image.dcm https://github.com/HamzaGbada/dicomPreProcess/blob/master/data/20587054.dcm
!wget -O image.png https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Steuben_-_Bataille_de_Poitiers.png/1280px-Steuben_-_Bataille_de_Poitiers.png

--2026-04-20 14:46:30--  https://github.com/HamzaGbada/dicomPreProcess/blob/master/data/20587054.dcm
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘image.dcm’

image.dcm               [ <=>                ] 215.91K  --.-KB/s    in 0.007s  

2026-04-20 14:46:31 (28.3 MB/s) - ‘image.dcm’ saved [221088]

--2026-04-20 14:46:31--  https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Steuben_-_Bataille_de_Poitiers.png/1280px-Steuben_-_Bataille_de_Poitiers.png
Resolving upload.wikimedia.org (upload.wikimedia.org)... 103.102.166.240, 2001:df2:e500:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|103.102.166.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2249885 (2.1M) [image/png]
Saving to: ‘image.png’

image.png           100%[===================>]   2.15M  --.-KB/s    in 0

In [2]:
import numpy as np
from medical_image import DicomImage, PNGImage, InMemoryImage, ImageExporter

## 1. Loading a DICOM Image

In [10]:
# Load a DICOM file
dicom = DicomImage("/content/image.dcm")
dicom.load()

print(f"Size: {dicom.width} x {dicom.height}")
print(f"Pixel data type: {dicom.pixel_data.dtype}")
print(f"Device: {dicom.device}")

Size: 2671 x 5101
Pixel data type: torch.uint16
Device: cpu


## 2. Loading a PNG Image

In [8]:
# Load a PNG file
png = PNGImage("image.png")
png.load()

print(f"Size: {png.width} x {png.height}")
print(f"Shape: {png.pixel_data.shape}")

Size: 4 x 1280
Shape: torch.Size([1053, 1280, 4])


## 3. Creating an Image from a NumPy Array

In [13]:
# Create a synthetic grayscale image
array = np.random.rand(256, 256).astype(np.float32)

img = DicomImage.from_array(array)
print(f"Size: {img.width} x {img.height}")
print(f"Pixel range: [{img.pixel_data.min():.2f}, {img.pixel_data.max():.2f}]")

Size: 256 x 256
Pixel range: [0.00, 1.00]


## 4. Cloning and Exporting

In [14]:
# Clone an image (deep copy of pixel data)
clone = img.clone()
print(f"Clone size: {clone.width} x {clone.height}")

# Export to PNG
# output_path = ImageExporter.save_as(img, format="PNG")
# print(f"Saved to: {output_path}")

Clone size: 256 x 256


## 5. Moving Images to GPU

In [15]:
import torch

if torch.cuda.is_available():
    img.to("cuda")
    print(f"Device: {img.device}")
    img.to("cpu")  # move back
else:
    print("CUDA not available, staying on CPU")

CUDA not available, staying on CPU


## 6. Annotations

In [16]:
from medical_image import Annotation, GeometryType

# Add a bounding box annotation
ann = Annotation(
    shape=GeometryType.RECTANGLE,
    coordinates=[50, 50, 150, 150],
    label="lesion",
)
img.add_annotation(ann)
print(f"Annotations: {len(img.annotations)}")
print(f"Label: {img.annotations[0].label}, Center: {img.annotations[0].center}")

Annotations: 1
Label: lesion, Center: (100.0, 100.0)


## 7. Patches and Region of Interest

In [17]:
from medical_image import PatchGrid, RegionOfInterest

# Split image into patches
grid = PatchGrid(img, patch_size=(64, 64))
print(f"Total patches: {len(grid.patches)}")
print(f"Grid shape: {len(grid.grid)} rows x {len(grid.grid[0])} cols")

# Access a single patch
patch = grid.patches[0]
print(f"Patch shape: {patch.pixel_data.shape}")

# Reconstruct the full image from patches
reconstructed = grid.to_image()
print(f"Reconstructed size: {reconstructed.width} x {reconstructed.height}")

Total patches: 16
Grid shape: 4 rows x 4 cols
Patch shape: torch.Size([64, 64])
Reconstructed size: 256 x 256


In [18]:
# Extract a region of interest by center coordinates
roi = RegionOfInterest.from_center(img, cx=128, cy=128, half_size=32)
roi_image = roi.load()
print(f"ROI size: {roi_image.width} x {roi_image.height}")

ROI size: 65 x 65
